# Notebook 08 — One-click validation and SASRec smoke test

Run this single notebook **after notebooks 01–07** for the Amazon Reviews 2023 `Beauty_and_Personal_Care` project.

It reuses the outputs created by the seven ETL notebooks. It does not redo the cleaning or split logic. It validates the frozen data, creates explicit split Parquet files, runs DuckDB checks, verifies federated-client readiness, and performs a small SASRec smoke test.

## Before running

1. Run notebooks `01_data_acquisition.ipynb` through `07_freeze_and_sanity_check.ipynb` in order.
2. Keep the same Google Drive project path for all notebooks.
3. Run every cell in this notebook from top to bottom.

This notebook is a connectivity/demo check. Its short SASRec metrics are not final research results.

In [1]:
# Install only the small packages needed by this notebook.
%pip -q install duckdb pyarrow


In [2]:
from pathlib import Path
import os, sys, json, hashlib, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A/04_Data/NewDataRunFiles')
else:
    BASE = Path(os.environ.get('RECSYS_BASE', './298A_recsys')).expanduser().resolve()

PROCESSED = BASE / 'processed'
REPORTS = BASE / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Project base:', BASE)


Mounted at /content/drive
Project base: /content/drive/MyDrive/298A/04_Data/NewDataRunFiles


In [3]:
# Confirm that notebooks 01-07 completed successfully.
required = [
    PROCESSED / 'dataset_info.json',
    PROCESSED / 'rec_eval.py',
    PROCESSED / 'manifest.json',
    PROCESSED / 'leakage_report.json',
    PROCESSED / 'interactions.parquet',
    PROCESSED / 'sequences.parquet',
    PROCESSED / 'items.parquet',
    PROCESSED / 'item_text.parquet',
    PROCESSED / 'clients' / 'clients.parquet',
    REPORTS / 'sanity_baselines.csv',
]
missing = [str(p.relative_to(BASE)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Run notebooks 01-07 first. Missing: ' + ', '.join(missing)
    )

info = json.loads((PROCESSED / 'dataset_info.json').read_text())
leakage = json.loads((PROCESSED / 'leakage_report.json').read_text())
assert leakage.get('all_passed') is True, leakage

interactions = pd.read_parquet(PROCESSED / 'interactions.parquet')
sequences = pd.read_parquet(PROCESSED / 'sequences.parquet')
clients = pd.read_parquet(PROCESSED / 'clients' / 'clients.parquet')

print('Dataset:', info['dataset'])
print('Users:', f"{info['n_users']:,}")
print('Items:', f"{info['n_items']:,}")
print('Interactions:', f"{len(interactions):,}")
print('Notebook 05 leakage checks: PASS')
display(interactions.head())


Dataset: Amazon Reviews 2023 - Beauty_and_Personal_Care
Users: 56,017
Items: 25,903
Interactions: 501,399
Notebook 05 leakage checks: PASS


,user_idx,item_idx,rating,timestamp,pos,split,row_id
0,0,10493,2.0,1570900999309,0,train,1013144
1,0,8934,2.0,1571890093750,1,train,1013145
2,0,6553,1.0,1571890171813,2,train,1013146
3,0,10027,2.0,1605156655266,3,train,1013147
4,0,16519,4.0,1605156954165,4,train,1013148


In [4]:
# Create explicit train/validation/test Parquet files.
# Notebook 05 remains the source of truth: its labels are train, val, and test.
SPLITS = PROCESSED / 'splits'
SPLITS.mkdir(parents=True, exist_ok=True)

for source_name, output_name in {
    'train': 'train.parquet',
    'val': 'validation.parquet',
    'test': 'test.parquet',
}.items():
    frame = interactions.loc[interactions['split'].eq(source_name)].copy()
    sort_cols = ['user_idx', 'timestamp'] + (['pos'] if 'pos' in frame.columns else [])
    frame = frame.sort_values(sort_cols).reset_index(drop=True)
    frame.to_parquet(SPLITS / output_name, index=False)
    print(f'{output_name}: {len(frame):,} rows, {frame.user_idx.nunique():,} users')

# Quality checks
required_i = {'user_idx', 'item_idx', 'rating', 'timestamp', 'split'}
required_s = {'user_idx', 'train_items', 'val_item', 'test_item', 'n_train'}
assert required_i.issubset(interactions.columns)
assert required_s.issubset(sequences.columns)
assert not interactions[list(required_i)].isna().any().any()
assert set(interactions.split.unique()) <= {'train', 'val', 'test'}
assert interactions.rating.between(1, 5).all()
assert (interactions.item_idx >= 1).all()
assert (interactions.user_idx >= 0).all()
assert not interactions.duplicated().any()

per_user = interactions.groupby(['user_idx', 'split']).size().unstack(fill_value=0)
assert (per_user[['train', 'val', 'test']] > 0).all().all()
assert per_user['val'].eq(1).all() and per_user['test'].eq(1).all()

ordered = interactions.sort_values(['user_idx', 'pos'])
for _, rows in ordered.groupby('user_idx', sort=False):
    train_ts = rows.loc[rows.split.eq('train'), 'timestamp']
    val_ts = rows.loc[rows.split.eq('val'), 'timestamp'].iloc[0]
    test_ts = rows.loc[rows.split.eq('test'), 'timestamp'].iloc[0]
    assert train_ts.max() <= val_ts <= test_ts

train_counts = interactions.query("split == 'train'").groupby('user_idx').size().sort_index()
sequence_counts = sequences.set_index('user_idx').n_train.sort_index()
assert sequence_counts.equals(train_counts.astype(sequence_counts.dtype))
assert len(clients) == interactions.user_idx.nunique()
assert clients.client_id.is_unique
assert set(clients.client_id) == set(interactions.user_idx)

quality = {
    'status': 'PASS',
    'users': int(interactions.user_idx.nunique()),
    'items': int(interactions.item_idx.nunique()),
    'interactions': int(len(interactions)),
    'checks': ['columns', 'nulls', 'ratings', 'IDs', 'duplicates',
               'chronology', 'sequence lengths', 'federated clients'],
}
(REPORTS / 'quality_gate.json').write_text(json.dumps(quality, indent=2))
print('Pandas quality gate: PASS')


train.parquet: 389,365 rows, 56,017 users
validation.parquet: 56,017 rows, 56,017 users
test.parquet: 56,017 rows, 56,017 users
Pandas quality gate: PASS


In [ ]:
# DuckDB quality checks over the explicit Parquet files
import duckdb

connection = duckdb.connect()
for name in ['train', 'validation', 'test']:
    path = (SPLITS / f'{name}.parquet').as_posix()
    row = connection.execute('''
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT user_idx) AS users,
               COUNT(DISTINCT item_idx) AS items,
               SUM(CASE WHEN user_idx IS NULL OR item_idx IS NULL THEN 1 ELSE 0 END) AS null_keys,
               SUM(CASE WHEN rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS invalid_ratings
        FROM read_parquet(?)
    ''', [path]).fetchone()
    print(name, row)
    assert row[0] > 0 and row[3] == 0 and row[4] == 0
connection.close()
print('DuckDB quality gate: PASS')


In [ ]:
# Load the shared evaluation module created by notebook 07
sys.path.insert(0, str(PROCESSED))
import rec_eval as R

data = R.load_dataset(PROCESSED)
baselines = pd.read_csv(REPORTS / 'sanity_baselines.csv')
print(f"Shared evaluator loaded: {data['n_users']:,} users, {data['n_items']:,} items")
display(baselines.round(4))


In [7]:
# Minimal SASRec-style Transformer for a short connectivity test
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

class TinySASRec(nn.Module):
    def __init__(self, n_items, max_len, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.item_embedding = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4*d_model,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def hidden(self, tokens):
        _, length = tokens.shape
        positions = torch.arange(length, device=tokens.device).unsqueeze(0)
        h = self.item_embedding(tokens) + self.position_embedding(positions)
        padding = tokens.eq(0)
        causal = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=tokens.device),
            diagonal=1,
        )
        return self.norm(self.encoder(
            h, mask=causal, src_key_padding_mask=padding
        ))

    def forward(self, tokens):
        return self.hidden(tokens) @ self.item_embedding.weight.T

    def next_item_scores(self, tokens):
        return self.forward(tokens)[:, -1, :]


In [8]:
# Train on a small subset for 2 epochs
X_train, Y_train = R.train_arrays(data['sequences'], data['max_seq_len'])
limit = min(4096, len(X_train))
X_train = X_train[:limit].astype(np.int64)
Y_train = Y_train[:limit].astype(np.int64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
model = TinySASRec(data['n_items'], data['max_seq_len']).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train)),
    batch_size=128, shuffle=True
)

for epoch in range(1, 3):
    model.train()
    losses = []
    for tokens, targets in loader:
        tokens, targets = tokens.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens)
        loss = F.cross_entropy(
            logits.reshape(-1, data['n_items'] + 1),
            targets.reshape(-1), ignore_index=0
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    print(f'epoch={epoch}, loss={np.mean(losses):.4f}')
print('SASRec smoke training: PASS on', device)


epoch=1, loss=29.7737
epoch=2, loss=23.2916
SASRec smoke training: PASS on cpu


In [9]:
# Evaluate a small fixed validation slice using the shared protocol
eval_users = np.arange(min(128, data['n_users']), dtype=np.int64)

@torch.no_grad()
def score_fn(inputs, users):
    del users
    model.eval()
    tokens = torch.as_tensor(inputs, dtype=torch.long, device=device)
    return model.next_item_scores(tokens).cpu().numpy().astype(np.float32)

result = R.evaluate(score_fn, data, split='val', sampled=True, users=eval_users)
print('Smoke-test metrics:')
display(pd.Series(result['metrics']).round(4).to_frame('value'))

smoke = {
    'status': 'PASS',
    'type': 'smoke_test_only',
    'device': str(device),
    'train_examples': int(limit),
    'validation_users': int(len(eval_users)),
    'metrics': result['metrics'],
    'warning': 'Do not report these short-run metrics as final results.',
}
(REPORTS / 'sasrec_smoke_test.json').write_text(json.dumps(smoke, indent=2))


Smoke-test metrics:


,value
Recall@5,0.9688
HitRate@5,0.9688
NDCG@5,1.6561
Recall@10,0.9688
HitRate@10,0.9688
NDCG@10,1.6561
Recall@20,0.9844
HitRate@20,0.9844
NDCG@20,1.6598
MRR,1.9388


508

In [10]:
# Final Model 2 readiness check
client_stats = json.loads((PROCESSED / 'clients' / 'client_stats.json').read_text())
print('Federated clients:', len(clients))
print('One client per user:', len(clients) == info['n_users'])
print('Median local training size:', client_stats['local_train_size']['50%'])
print('Local-size Gini:', client_stats['local_size_gini'])
print('Non-IID category field:', client_stats['category_level_for_non_iid'])

print('\nSUCCESSFUL OUTPUTS')
for path in [
    PROCESSED / 'splits' / 'train.parquet',
    PROCESSED / 'splits' / 'validation.parquet',
    PROCESSED / 'splits' / 'test.parquet',
    REPORTS / 'quality_gate.json',
    REPORTS / 'sasrec_smoke_test.json',
]:
    print(path.relative_to(BASE), 'exists =', path.exists())


Federated clients: 56017
One client per user: True
Median local training size: 4.0
Local-size Gini: 0.4466
Non-IID category field: cat_l2

SUCCESSFUL OUTPUTS
processed/splits/train.parquet exists = True
processed/splits/validation.parquet exists = True
processed/splits/test.parquet exists = True
reports/quality_gate.json exists = True
reports/sasrec_smoke_test.json exists = True


## What to show the team

Show the successful output cells in this order:

`Notebook 07 frozen artifacts → explicit Parquet splits → DuckDB PASS → SASRec loss decreases → SASRec metrics → one client per user`

After this works, implement full centralized SASRec first. Then implement Model 2 using the existing `sequences.parquet`, `clients/clients.parquet`, and shared `rec_eval.py`; do not create a new split inside the model notebook.